In [3]:
import os
import re
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import glob

In [3]:
def _infer_horizon_from_target(target_col):
    return int(re.search(r"_h(\d+)$", target_col).group(1))

In [4]:
def walk_forward_oos(
    df,
    target_col,
    model_factory,
    feature_cols=None,
    train_size=252 * 5,
    test_size=63,
    step_size=None,
    horizon=None,
    expanding=False,
    save_path=None,
):
    if horizon is None:
        horizon = _infer_horizon_from_target(target_col)

    if feature_cols is None:
        feature_cols = [c for c in df.columns if not c.startswith("target_")]

    data = df[feature_cols + [target_col]].copy()
    n = len(data)

    gap = horizon - 1

    if step_size is None:
        step_size = test_size

    preds = []
    folds = []

    fold_id = 0
    test_start = train_size + gap

    while test_start < n:
        test_end = min(test_start + test_size, n)
        train_end = test_start - gap
        train_start = 0 if expanding else train_end - train_size

        train_idx = data.index[train_start:train_end]
        test_idx = data.index[test_start:test_end]

        X_train = data.loc[train_idx, feature_cols]
        y_train = data.loc[train_idx, target_col]
        X_test = data.loc[test_idx, feature_cols]
        y_test = data.loc[test_idx, target_col]

        model = model_factory()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        preds.append(
            pd.DataFrame(
                {
                    "y_true": y_test.values,
                    "y_pred": np.asarray(y_pred).reshape(-1),
                    "fold": fold_id,
                },
                index=test_idx,
            )
        )

        folds.append(
            {
                "fold": fold_id,
                "train_start": train_idx[0],
                "train_end": train_idx[-1],
                "test_start": test_idx[0],
                "test_end": test_idx[-1],
            }
        )

        fold_id += 1
        test_start += step_size

    pred_df = pd.concat(preds).sort_index()
    folds_df = pd.DataFrame(folds)

    if save_path is not None:
        pred_df.to_parquet(save_path)

    return pred_df, folds_df

In [4]:
def _loss_by_objective(y_true, y_pred, objective, params=None):
    params = params or {}
    e = y_true - y_pred

    if objective in ["regression", "regression_l2"]:
        return np.mean(e ** 2)

    if objective == "regression_l1":
        return np.mean(np.abs(e))

    if objective == "huber":
        delta = params.get("alpha", 1.0)
        ae = np.abs(e)
        return np.mean(np.where(ae <= delta, 0.5 * e ** 2, delta * (ae - 0.5 * delta)))

    if objective == "fair":
        c = params.get("fair_c", 1.0)
        ae = np.abs(e)
        return np.mean(c ** 2 * (ae / c - np.log1p(ae / c)))

    if objective == "quantile":
        alpha = params.get("alpha", 0.5)
        return np.mean(np.maximum(alpha * e, (alpha - 1) * e))

    if objective == "mape":
        denom = np.where(np.abs(y_true) < 1e-8, np.nan, y_true)
        return np.nanmean(np.abs(e / denom))

    return np.mean(e ** 2)

In [24]:
def tune_lgbm_once(
    X,
    y,
    objective="regression",
    n_trials=50,
    random_seed=5,
    val_size=0.2,
):
    n_val = int(len(X) * val_size)

    X_tr = X.iloc[:-n_val]
    y_tr = y.iloc[:-n_val]
    X_val = X.iloc[-n_val:]
    y_val = y.iloc[-n_val:]

    def optuna_objective(trial):
        params = {
            "objective": objective,
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 7, 63),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": random_seed,
            "n_jobs": -1,
            "verbosity": -1,
        }

        if objective in ["huber", "quantile"]:
            params["alpha"] = trial.suggest_float("alpha", 0.50, 0.95)

        if objective == "fair":
            params["fair_c"] = trial.suggest_float("fair_c", 0.1, 10.0, log=True)

        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)

        return _loss_by_objective(y_val.values, pred, objective)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=random_seed),
    )

    study.optimize(optuna_objective, n_trials=n_trials, show_progress_bar=False)

    return {
        **study.best_params,
        "objective": objective,
        "random_state": random_seed,
        "n_jobs": -1,
        "verbosity": -1,
    }

In [25]:
def make_lgbm_fixed(params):
    return LGBMRegressor(**params)

In [28]:
def run_lgbm_oos_forecasts(
    features_targets,
    objectives=("regression", "regression_l1", "huber", "fair", "quantile", "mape"),
    n_trials=50,
    random_seed=5,
    train_size=252 * 5,
    test_size=63,
    step_size=63,
    expanding=False,
):
    out = {}

    for asset, df in features_targets.items():
        target_cols = [c for c in df.columns if c.startswith("target_")]
        feature_cols = [c for c in df.columns if not c.startswith("target_")]

        for target_col in target_cols:
            h = _infer_horizon_from_target(target_col)

            X_tune = df[feature_cols].iloc[:train_size]
            y_tune = df[target_col].iloc[:train_size]

            for objective in objectives:
                key = f"{asset}_{target_col}_{objective}"
                save_path = f"forecasts/{key}.parquet"

                best_params = tune_lgbm_once(
                    X=X_tune,
                    y=y_tune,
                    objective=objective,
                    n_trials=n_trials,
                    random_seed=random_seed,
                    val_size=0.2,
                )

                model_factory = lambda best_params=best_params: make_lgbm_fixed(best_params)

                pred_df, folds_df = walk_forward_oos(
                    df=df,
                    target_col=target_col,
                    model_factory=model_factory,
                    feature_cols=feature_cols,
                    train_size=train_size,
                    test_size=test_size,
                    step_size=step_size,
                    horizon=h,
                    expanding=expanding,
                    save_path=save_path,
                )

                out[key] = {
                    "preds": pred_df,
                    "folds": folds_df,
                    "path": save_path,
                    "best_params": best_params,
                }

    return out

In [21]:
files = sorted(glob.glob("data/*.parquet"))

In [29]:
for file in files:
    df = pd.read_parquet(file)

    target_col = [c for c in df.columns if c.startswith("target_")][0]
    feature_cols = [c for c in df.columns if not c.startswith("target_")]

    asset = file.split("/")[-1].replace(".parquet", "")
    h = _infer_horizon_from_target(target_col)

    X_tune = df[feature_cols].iloc[:252 * 5]
    y_tune = df[target_col].iloc[:252 * 5]

    for objective in ("regression", "regression_l1", "huber", "fair", "quantile", "mape"):

        best_params = tune_lgbm_once(
            X=X_tune,
            y=y_tune,
            objective=objective,
            n_trials=50,
            random_seed=5,
            val_size=0.2,
        )

        save_path = f"forecasts/{asset}_{objective}.parquet"

        model_factory = lambda best_params=best_params: make_lgbm_fixed(best_params)

        pred_df, folds_df = walk_forward_oos(
            df=df,
            target_col=target_col,
            model_factory=model_factory,
            feature_cols=feature_cols,
            train_size=252 * 5,
            test_size=63,
            step_size=63,
            horizon=h,
            expanding=False,
            save_path=save_path,
        )

        print(save_path)

[I 2026-04-24 19:47:59,253] A new study created in memory with name: no-name-16987932-05fb-45bc-94f3-f2bba108c434
[I 2026-04-24 19:47:59,522] Trial 0 finished with value: 0.0005142048399590846 and parameters: {'n_estimators': 300, 'learning_rate': 0.06789203913136307, 'num_leaves': 18, 'max_depth': 8, 'min_child_samples': 54, 'subsample': 0.8058719314513229, 'colsample_bytree': 0.8829539282401577, 'reg_alpha': 0.00046319289692831295, 'reg_lambda': 4.690342036615963e-06}. Best is trial 0 with value: 0.0005142048399590846.
[I 2026-04-24 19:47:59,708] Trial 1 finished with value: 0.00041413599622049254 and parameters: {'n_estimators': 269, 'learning_rate': 0.006368201795811424, 'num_leaves': 49, 'max_depth': 5, 'min_child_samples': 24, 'subsample': 0.9399685156006394, 'colsample_bytree': 0.6370432309961123, 'reg_alpha': 5.347061407655772e-05, 'reg_lambda': 4.6208236528276254e-06}. Best is trial 1 with value: 0.00041413599622049254.
[I 2026-04-24 19:47:59,952] Trial 2 finished with value: 

forecasts/copper_target_copper_h1_regression.parquet


[I 2026-04-24 19:48:17,138] Trial 0 finished with value: 0.016084671710393753 and parameters: {'n_estimators': 300, 'learning_rate': 0.06789203913136307, 'num_leaves': 18, 'max_depth': 8, 'min_child_samples': 54, 'subsample': 0.8058719314513229, 'colsample_bytree': 0.8829539282401577, 'reg_alpha': 0.00046319289692831295, 'reg_lambda': 4.690342036615963e-06}. Best is trial 0 with value: 0.016084671710393753.
[I 2026-04-24 19:48:17,391] Trial 1 finished with value: 0.015289623154387236 and parameters: {'n_estimators': 269, 'learning_rate': 0.006368201795811424, 'num_leaves': 49, 'max_depth': 5, 'min_child_samples': 24, 'subsample': 0.9399685156006394, 'colsample_bytree': 0.6370432309961123, 'reg_alpha': 5.347061407655772e-05, 'reg_lambda': 4.6208236528276254e-06}. Best is trial 1 with value: 0.015289623154387236.
[I 2026-04-24 19:48:17,669] Trial 2 finished with value: 0.01580342134750708 and parameters: {'n_estimators': 666, 'learning_rate': 0.028402488195465363, 'num_leaves': 41, 'max_

forecasts/copper_target_copper_h1_regression_l1.parquet


[I 2026-04-24 19:48:54,419] Trial 0 finished with value: 0.0002571024199795423 and parameters: {'n_estimators': 300, 'learning_rate': 0.06789203913136307, 'num_leaves': 18, 'max_depth': 8, 'min_child_samples': 54, 'subsample': 0.8058719314513229, 'colsample_bytree': 0.8829539282401577, 'reg_alpha': 0.00046319289692831295, 'reg_lambda': 4.690342036615963e-06, 'alpha': 0.5844745528975632}. Best is trial 0 with value: 0.0002571024199795423.
[I 2026-04-24 19:48:54,478] Trial 1 finished with value: 0.00020688518765468218 and parameters: {'n_estimators': 172, 'learning_rate': 0.04567756872397172, 'num_leaves': 32, 'max_depth': 3, 'min_child_samples': 90, 'subsample': 0.6370432309961123, 'colsample_bytree': 0.7071175095405257, 'reg_alpha': 4.6208236528276254e-06, 'reg_lambda': 0.004561326707315878, 'alpha': 0.7609270145852952}. Best is trial 1 with value: 0.00020688518765468218.
[I 2026-04-24 19:48:54,527] Trial 2 finished with value: 0.00020394106207082297 and parameters: {'n_estimators': 64

...

forecasts/wheat_target_wheat_h5_fair.parquet


[I 2026-04-24 20:59:45,756] Trial 0 finished with value: 0.02736665534596067 and parameters: {'n_estimators': 300, 'learning_rate': 0.06789203913136307, 'num_leaves': 18, 'max_depth': 8, 'min_child_samples': 54, 'subsample': 0.8058719314513229, 'colsample_bytree': 0.8829539282401577, 'reg_alpha': 0.00046319289692831295, 'reg_lambda': 4.690342036615963e-06, 'alpha': 0.5844745528975632}. Best is trial 0 with value: 0.02736665534596067.
[I 2026-04-24 20:59:45,808] Trial 1 finished with value: 0.021365815300043986 and parameters: {'n_estimators': 172, 'learning_rate': 0.04567756872397172, 'num_leaves': 32, 'max_depth': 3, 'min_child_samples': 90, 'subsample': 0.6370432309961123, 'colsample_bytree': 0.7071175095405257, 'reg_alpha': 4.6208236528276254e-06, 'reg_lambda': 0.004561326707315878, 'alpha': 0.7609270145852952}. Best is trial 1 with value: 0.021365815300043986.
[I 2026-04-24 20:59:46,083] Trial 2 finished with value: 0.02592490541045983 and parameters: {'n_estimators': 640, 'learnin

forecasts/wheat_target_wheat_h5_quantile.parquet


/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_35859/2801153681.py:25: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs(e / y_true))
[I 2026-04-24 21:00:27,178] Trial 0 finished with value: inf and parameters: {'n_estimators': 300, 'learning_rate': 0.06789203913136307, 'num_leaves': 18, 'max_depth': 8, 'min_child_samples': 54, 'subsample': 0.8058719314513229, 'colsample_bytree': 0.8829539282401577, 'reg_alpha': 0.00046319289692831295, 'reg_lambda': 4.690342036615963e-06}. Best is trial 0 with value: inf.
/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_35859/2801153681.py:25: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs(e / y_true))
[I 2026-04-24 21:00:27,450] Trial 1 finished with value: inf and parameters: {'n_estimators': 269, 'learning_rate': 0.006368201795811424, 'num_leaves': 49, 'max_depth': 5, 'min_child_samples': 24, 'subsample': 0.9399685156006394, 'colsample_bytree': 0.6370432309961123, 'reg_al

forecasts/wheat_target_wheat_h5_mape.parquet
